# Evaluate & Compare Models

Evaluate fine-tuned RT-DETR models, compare hyperparameter tuning runs, and select the best configuration.

**Sections:**
1. Evaluate a single model (mAP metrics)
2. Visualize predictions on sample images
3. Compare training runs (loss curves, final metrics)
4. HP tuning strategy & best model selection

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from datasets import load_dataset
from transformers import RTDetrForObjectDetection

from moku.dataset import CATEGORIES, ID_TO_CATEGORY
from moku.training import (
    HF_DATASET,
    HF_MODEL,
    collate_fn,
    evaluate_map,
    format_map_results,
    load_image_processor,
    load_training_runs,
    make_eval_transform,
    summarize_runs,
)
from moku.viz import render_sample

## Load Dataset & Model

Load the test split and a trained model checkpoint. Change `MODEL_PATH` to evaluate different runs.

In [ ]:
# Point to a local checkpoint or HF Hub model
MODEL_PATH = "runs/baseline"  # or HF_MODEL for the pushed model

dataset = load_dataset(HF_DATASET)
image_processor = load_image_processor()

model = RTDetrForObjectDetection.from_pretrained(
    MODEL_PATH,
    num_labels=len(CATEGORIES),
    id2label=ID_TO_CATEGORY,
    label2id=CATEGORIES,
)

dataset["test"].set_transform(make_eval_transform(image_processor))
print(f"Model loaded from: {MODEL_PATH}")
print(f"Test set: {len(dataset['test'])} images")

## Compute mAP Metrics

In [ ]:
metrics = evaluate_map(
    model=model,
    dataset=dataset["test"],
    image_processor=image_processor,
    batch_size=8,
    threshold=0.3,
)

display(format_map_results(metrics))

## Visualize Predictions

Show model predictions on sample test images with bounding boxes and confidence scores.

In [ ]:
# Load raw test images (without transforms) for visualization
raw_test = load_dataset(HF_DATASET, split="test")

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
model_viz = model.to(device)
model_viz.eval()

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for idx, ax in enumerate(axes.flat):
    if idx >= len(raw_test):
        ax.axis("off")
        continue

    sample = raw_test[idx]
    image = sample["image"].convert("RGB")

    # Run inference
    inputs = image_processor(images=[image], return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model_viz(**inputs)

    # Post-process
    target_size = torch.tensor([[image.height, image.width]])
    results = image_processor.post_process_object_detection(
        outputs, target_sizes=target_size, threshold=0.5
    )[0]

    # Draw predictions
    ax.imshow(image)
    for box, score, label in zip(results["boxes"], results["scores"], results["labels"]):
        x1, y1, x2, y2 = box.cpu().tolist()
        name = ID_TO_CATEGORY[label.item()]
        color = {"board": "#d95f02", "black_stone": "#e7298a", "white_stone": "#1b9e77"}[name]
        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=1.5, edgecolor=color, facecolor="none")
        ax.add_patch(rect)
        ax.text(x1, y1 - 2, f"{name} {score:.2f}", fontsize=7, color=color, weight="bold")
    ax.set_title(f"Test image {idx}")
    ax.axis("off")

plt.tight_layout()
plt.show()

## Compare Training Runs

Load training logs from all runs in the `runs/` directory and compare loss curves.

Each run is stored as a sub-directory (e.g., `runs/baseline/`, `runs/lr_5e-5/`, etc.) created by the Trainer in `02_Train_Model.ipynb`. Change `RUN_NAME` and `OUTPUT_DIR` there to create new runs with different hyperparameters.

In [ ]:
RUNS_DIR = Path("runs")

# Load all training logs
logs = load_training_runs(RUNS_DIR)
if logs.empty:
    print("No training runs found in runs/. Train a model first with 02_Train_Model.ipynb.")
else:
    runs = logs["run"].unique()
    print(f"Found {len(runs)} run(s): {', '.join(runs)}")

    # Plot training loss
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for run_name in runs:
        run_logs = logs[logs["run"] == run_name]
        train_logs = run_logs.dropna(subset=["loss"])
        eval_logs = run_logs.dropna(subset=["eval_loss"])
        axes[0].plot(train_logs["step"], train_logs["loss"], label=run_name, alpha=0.8)
        if not eval_logs.empty:
            axes[1].plot(eval_logs["step"], eval_logs["eval_loss"], label=run_name, marker="o", markersize=3)

    axes[0].set_xlabel("Step")
    axes[0].set_ylabel("Training Loss")
    axes[0].set_title("Training Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].set_xlabel("Step")
    axes[1].set_ylabel("Eval Loss")
    axes[1].set_title("Validation Loss")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## Evaluate All Runs (mAP)

Compute mAP for the best checkpoint of each run and rank them.

In [ ]:
all_results = []

for run_dir in sorted(RUNS_DIR.iterdir()):
    if not run_dir.is_dir():
        continue
    # Find best checkpoint (or use the run dir itself if it has model files)
    config_file = run_dir / "config.json"
    if not config_file.exists():
        # Try the latest checkpoint sub-directory
        checkpoints = sorted(run_dir.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[1]))
        if checkpoints:
            run_dir = checkpoints[-1]
            config_file = run_dir / "config.json"
    if not config_file.exists():
        continue

    print(f"Evaluating: {run_dir.name}...")
    run_model = RTDetrForObjectDetection.from_pretrained(
        str(run_dir), num_labels=len(CATEGORIES), id2label=ID_TO_CATEGORY, label2id=CATEGORIES
    )

    run_metrics = evaluate_map(
        model=run_model,
        dataset=dataset["test"],
        image_processor=image_processor,
        batch_size=8,
    )
    all_results.append({"run": run_dir.parent.name, "checkpoint": run_dir.name, **run_metrics})

if all_results:
    results_df = pd.DataFrame(all_results)[["run", "checkpoint", "map", "map_50", "map_75", "mar_100"]]
    results_df = results_df.sort_values("map", ascending=False).reset_index(drop=True)
    display(results_df)

    best = results_df.iloc[0]
    print(f"\nBest run: {best['run']} (mAP={best['map']:.4f}, mAP@50={best['map_50']:.4f})")
else:
    print("No trained models found in runs/. Train a model first.")

## HP Tuning Strategy

**Do we need hyperparameter tuning?** Yes — with ~492 images, the model is sensitive to learning rate and regularization. A wrong LR easily leads to divergence or overfitting.

**What to tune** (in priority order):

| Parameter | Range | Impact |
|-----------|-------|--------|
| `learning_rate` | `[5e-5, 1e-4, 2e-4, 5e-4]` | Critical — most impactful HP |
| `weight_decay` | `[1e-4, 1e-3]` | Moderate — prevents overfitting on small dataset |
| `num_train_epochs` | `[30, 50, 100]` | Use early stopping via `load_best_model_at_end` |

**Recommended approach:**
- Run 6–8 configs (grid on LR × WD) — manageable locally or on a single GPU
- Use `eval_loss` plateau to detect overfitting
- Compare runs in this notebook

**For HF Jobs / remote GPU:**
1. Change `RUN_NAME` and HP values in `02_Train_Model.ipynb`
2. Run on a GPU instance (HF Spaces, cloud VM, etc.)
3. Download `runs/<name>/` results locally
4. Re-run this notebook to compare all runs

**Suggested configs for a full sweep:**
```python
configs = [
    {"name": "lr_5e-5_wd_1e-4", "learning_rate": 5e-5, "weight_decay": 1e-4},
    {"name": "lr_1e-4_wd_1e-4", "learning_rate": 1e-4, "weight_decay": 1e-4},
    {"name": "lr_2e-4_wd_1e-4", "learning_rate": 2e-4, "weight_decay": 1e-4},
    {"name": "lr_5e-4_wd_1e-4", "learning_rate": 5e-4, "weight_decay": 1e-4},
    {"name": "lr_5e-5_wd_1e-3", "learning_rate": 5e-5, "weight_decay": 1e-3},
    {"name": "lr_1e-4_wd_1e-3", "learning_rate": 1e-4, "weight_decay": 1e-3},
    {"name": "lr_2e-4_wd_1e-3", "learning_rate": 2e-4, "weight_decay": 1e-3},
    {"name": "lr_5e-4_wd_1e-3", "learning_rate": 5e-4, "weight_decay": 1e-3},
]
```